# FinSight — Behavioral Finance Synthetic Data Generator

Notebook ini membangkitkan data sintetis mutasi rekening untuk melatih model anomaly detection.
Data dikontrol oleh distribusi probabilitas dan psikologi keuangan (Behavioral Finance), bukan random biasa.

## Rules Engine

1. **Frekuensi (Poisson):** Jumlah transaksi harian per kategori diatur `lambda` per persona.
2. **Dynamic Throttling:** Rem otomatis berdasarkan rasio `sisa_saldo / gaji` — Sultan (>70%), Panik (15–70%), Survival (<15%).
3. **Cause-and-Effect Persona:** 15% nasabah `is_dynamic`; persona turun jika survival streak >= 7 hari.
4. **Peak Season & Windfall:** Multiplier di tgl 1–5, tanggal kembar, dan 2% peluang/hari pemasukan dadakan.
5. **Injeksi Noise:** Fee admin acak 30%, anomali markup 3x–8x (2%), typo NLP pada catatan P2P (15%).

## Skema DataFrame

**df_nasabah:** `id_user`, `nama_nasabah`, `tanggal_lahir`, `nama_ibu_kandung`, `segmen_demografi`, `gaji_bulanan`, `persona_dasar`, `is_dynamic`, `persona_bulan_1/2/3`

**df_transaksi:** `id_transaksi`, `id_user`, `timestamp`, `tipe_mutasi`, `deskripsi_mutasi`, `catatan_mutasi`, `mcc`, `nominal`, `sisa_saldo`, `kategori_besar`, `kategori_detail`, `label_anomali`, `gt_kategori_besar`, `gt_kategori_detail`

In [1]:
import pandas as pd
import numpy as np
!pip install faker
from faker import Faker
import random
from datetime import datetime, timedelta

# Profil Nasabah

In [2]:
!pip install faker
from faker import Faker 
import random
import numpy as np 
import os

OUTPUT_DIR = '../../../data/new'
os.makedirs(OUTPUT_DIR, exist_ok=True)
fake = Faker('id_ID')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

print("Membangkitkan data nasabah...")

jumlah_nasabah = 500
data_nasabah = []

for i in range(jumlah_nasabah):
    id_user = f"USR-{str(i+1).zfill(3)}"
    nama = fake.name()
    tanggal_lahir = fake.date_of_birth(minimum_age=18, maximum_age=35).strftime('%Y-%m-%d')
    nama_ibu_kandung = fake.name_female()

    if i < 200: # 40% Mahasiswa (200 orang)
        segmen = 'Mahasiswa'
        pendapatan = random.randint(150, 350) * 10000 # Rp 1.5 Juta - Rp 3.5 Juta
    elif i < 400: # 40% First Jobber (200 orang)
        segmen = 'First Jobber'
        pendapatan = random.randint(450, 600) * 10000 # Rp 4.5 Juta - Rp 6.0 Juta
    else: # 20% Profesional (100 orang)
        segmen = 'Profesional'
        pendapatan = random.randint(800, 2000) * 10000 
        
    base_persona = random.choice(['Spendthrift', 'Unconflicted', 'Tightwad'])

    # 15% nasabah is_dynamic
    is_dynamic = 1 if random.random() < 0.15 else 0

    data_nasabah.append([
        id_user, nama, tanggal_lahir, nama_ibu_kandung, segmen, pendapatan,
        base_persona, is_dynamic,
        base_persona, base_persona, base_persona 
    ])


df_nasabah = pd.DataFrame(data_nasabah, columns=[
    'id_user', 'nama_nasabah', 'tanggal_lahir', 'nama_ibu_kandung',
    'segmen_demografi', 'gaji_bulanan', 'persona_dasar', 'is_dynamic',
    'persona_bulan_1', 'persona_bulan_2', 'persona_bulan_3' 
])


df_nasabah = df_nasabah.sample(frac=1, random_state=42).reset_index(drop=True)

print("df_nasabah selesai.\n")
display(df_nasabah.head(20))

print("\n RANGKUMAN DEMOGRAFI:")
print(df_nasabah['segmen_demografi'].value_counts())
print("\n RANGKUMAN PERSONA DINAMIS (1 = Bisa Berubah, 0 = Statis):")
print(df_nasabah['is_dynamic'].value_counts())

Membangkitkan data nasabah...
df_nasabah selesai.



,id_user,nama_nasabah,tanggal_lahir,nama_ibu_kandung,segmen_demografi,gaji_bulanan,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
0,USR-362,Sutan Perkasa Hakim,1992-11-25,Gina Permata,First Jobber,4750000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
1,USR-074,"Bakiadi Suryono, M.TI.",2002-04-30,dr. Cornelia Prakasa,Mahasiswa,3210000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
2,USR-375,Karen Mulyani,2006-04-09,Cinthia Halimah,First Jobber,5580000,Tightwad,0,Tightwad,Tightwad,Tightwad
3,USR-156,Sari Laksita,2006-01-30,Lasmanto Hidayat,Mahasiswa,2970000,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
4,USR-105,"drg. Jaka Fujiati, M.Farm",1995-03-18,"R.A. Yance Irawan, S.T.",Mahasiswa,1600000,Tightwad,1,Tightwad,Tightwad,Tightwad
5,USR-395,R.A. Qori Wibisono,2002-07-26,Tgk. Harjo Mandala,First Jobber,4690000,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
6,USR-378,Eva Napitupulu,1996-04-25,Qori Prasasta,First Jobber,5960000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
7,USR-125,Salimah Hutapea,1993-03-16,Kusuma Hutagalung,Mahasiswa,2140000,Spendthrift,1,Spendthrift,Spendthrift,Spendthrift
8,USR-069,Salimah Maheswara,2003-05-03,Ophelia Saragih,Mahasiswa,1550000,Tightwad,0,Tightwad,Tightwad,Tightwad
9,USR-451,"KH. Hendra Halim, S.T.",2005-12-26,Nardi Mustofa,Profesional,14400000,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted



 RANGKUMAN DEMOGRAFI:
segmen_demografi
First Jobber    200
Mahasiswa       200
Profesional     100
Name: count, dtype: int64

 RANGKUMAN PERSONA DINAMIS (1 = Bisa Berubah, 0 = Statis):
is_dynamic
0    421
1     79
Name: count, dtype: int64


In [3]:
print("Jumlah Persona per Bulan 1:")
print(df_nasabah['persona_bulan_1'].value_counts())

print("\nJumlah Persona per Bulan 2:")
print(df_nasabah['persona_bulan_2'].value_counts())

print("\nJumlah Persona per Bulan 3:")
print(df_nasabah['persona_bulan_3'].value_counts())

Jumlah Persona per Bulan 1:
persona_bulan_1
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64

Jumlah Persona per Bulan 2:
persona_bulan_2
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64

Jumlah Persona per Bulan 3:
persona_bulan_3
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64


In [4]:
df_nasabah.to_csv(f'{OUTPUT_DIR}/df_nasabah.csv', index=False)
print(f"df_nasabah final (termasuk persona bulanan) berhasil disimpan sebagai '{OUTPUT_DIR}/df_nasabah.csv'")

df_nasabah final (termasuk persona bulanan) berhasil disimpan sebagai '../../../data/new/df_nasabah.csv'


# Tahap 2 — Master Dictionary Merchant

## Master Merchant Dictionary

`master_merchant` adalah nested dict tiga level: **Kategori Besar → Kategori Detail → Merchant**.
Setiap merchant punya `mu` (harga rata-rata), `sigma` (deviasi), dan `mcc` (4-digit MCC).
Tersedia merchant kelas atas dan bawah per kategori agar K-Medoids bisa memisahkan persona.

## Arsitektur Klasifikasi MCC

**Jalur 1 — Rule-Based (MCC bank)**

| Kategori Besar | Kategori Detail | MCC |
| :--- | :--- | :--- |
| Needs | Tagihan & Utilitas | `4900` |
| Needs | Transportasi | `4121`, `5541` |
| Needs | Groceries & Kebutuhan Pokok | `5411` |
| Needs | Kesehatan & Perawatan Diri | `5912`, `8099` |
| Wants | F&B dan Nongkrong | `5812`, `5814` |
| Wants | Hiburan & Langganan | `4899`, `7832`, `7999` |
| Wants | Belanja Online & Fashion | `5311`, `5651` |
| Wants | Produktivitas & Digital | `5734` |
| Transfer | Transfer P2P | `4829` |
| Transfer | Investasi & Finansial | `6211`, `6012` |

**Jalur 2 — NLP Target (Transfer P2P, MCC 4829)**

Model TF-IDF + Naive Bayes membaca `catatan_mutasi` untuk mengklasifikasikan tujuan transfer (Needs / Wants / Investasi).

## Tahap 2 — Running Syntax

In [5]:
print("Membangun master merchant dictionary...")

master_merchant = {
    'Needs': {
        'Tagihan & Utilitas': {
            'PLN MOBILE'           : {'mu': 200000,  'sigma': 50000,   'mcc': '4900', 'w': [5, 5, 5]},
            'PDAM'                 : {'mu': 100000,  'sigma': 25000,   'mcc': '4900', 'w': [4, 4, 4]},
            'INDIHOME'             : {'mu': 350000,  'sigma': 20000,   'mcc': '4900', 'w': [2, 3, 1]},
            'FIRSTMEDIA'           : {'mu': 400000,  'sigma': 30000,   'mcc': '4900', 'w': [1, 2, 0]},
            'UKT KAMPUS'           : {'mu': 5000000, 'sigma': 1000000, 'mcc': '8220', 'w': [3, 3, 3]},
            'KPR BANK'             : {'mu': 3000000, 'sigma': 750000,  'mcc': '6012', 'w': [0, 2, 0]},
            'CICILAN KTA'          : {'mu': 1500000, 'sigma': 300000,  'mcc': '6012', 'w': [0, 2, 0]},
            'PAKET DATA TELKOMSEL' : {'mu': 55000,   'sigma': 10000,   'mcc': '4814', 'w': [6, 4, 3]},
            'PAKET DATA XL/AXIS'   : {'mu': 30000,   'sigma': 10000,   'mcc': '4814', 'w': [4, 3, 5]},
            'PAKET DATA INDOSAT'   : {'mu': 25000,   'sigma': 8000,    'mcc': '4814', 'w': [4, 3, 5]},
            'BIAYA KOSAN BULANAN'  : {'mu': 800000, 'sigma': 300000,  'mcc': '6513', 'w': [6, 4, 4]},
            'KONTRAKAN/SEWA RUMAH' : {'mu': 1200000, 'sigma': 500000,  'mcc': '6513', 'w': [3, 3, 2]},
            'LAUNDRY KILOAN'       : {'mu': 35000,   'sigma': 10000,   'mcc': '7210', 'w': [6, 3, 0]},
            'ICLOUD/GOOGLE ONE'    : {'mu': 45000,   'sigma': 0,       'mcc': '5734', 'w': [4, 3, 0]},
        },
        'Transportasi': {
            'SPBU PERTAMINA'  : {'mu': 70000,  'sigma': 50000,  'mcc': '5541', 'w': [5, 4, 3]},
            'SPBU SHELL'      : {'mu': 70000,  'sigma': 40000,  'mcc': '5541', 'w': [4, 2, 0]},
            'SPBU VIVO'       : {'mu': 70000,   'sigma': 30000,  'mcc': '5541', 'w': [2, 3, 5]},
            'GOJEK INDONESIA' : {
                'mu': 20000, 'sigma': 15000, 'mcc': '4121', 'w': [4, 5, 3],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'Transportasi'), 
                    'First Jobber': ('Needs', 'Transportasi'),  
                    'Profesional' : ('Needs', 'Transportasi'), 
                },
            },
            'GRAB INDONESIA'  : {
                'mu': 20000, 'sigma': 15000, 'mcc': '4121', 'w': [4, 5, 2],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'Transportasi'),
                    'First Jobber': ('Needs', 'Transportasi'),
                    'Profesional' : ('Needs', 'Transportasi'),
                },
            },
            'MAXIM INDONESIA' : {
                'mu': 20000, 'sigma': 8000, 'mcc': '4121', 'w': [1, 3, 6],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'Transportasi'),
                    'First Jobber': ('Needs', 'Transportasi'),  
                    'Profesional' : ('Needs', 'Transportasi'),
                },
            },
            'INDRIVER'        : {
                'mu': 15000, 'sigma': 7000, 'mcc': '4121', 'w': [1, 3, 6],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'Transportasi'),
                    'First Jobber': ('Needs', 'Transportasi'),
                    'Profesional' : ('Needs', 'Transportasi'),
                },
            },
            'TRANSJAKARTA/BRT'  : {'mu': 3500,    'sigma': 500,    'mcc': '4111', 'w': [0, 2, 7]},
            'KRL COMMUTER LINE' : {'mu': 10000,   'sigma': 20000,  'mcc': '4111', 'w': [1, 3, 7]},
            'BIS ANTAR KOTA'    : {'mu': 60000,   'sigma': 30000,  'mcc': '4111', 'w': [2, 3, 4]},
            'PARKIR'            : {'mu': 4000,    'sigma': 4000,   'mcc': '4121', 'w': [5, 4, 2]},
            'E-TOLL FLAZZ'      : {'mu': 20000,  'sigma': 50000,  'mcc': '4784', 'w': [5, 3, 1]},
            'BENGKEL MOTOR'     : {'mu': 80000,  'sigma': 50000,  'mcc': '7538', 'w': [3, 5, 5]},
            'BENGKEL MOBIL'     : {'mu': 200000,  'sigma': 150000, 'mcc': '7538', 'w': [4, 2, 0]},
            'CUCI MOTOR/MOBIL'  : {'mu': 25000,   'sigma': 5000,   'mcc': '7542', 'w': [4, 3, 2]},
        },

        'Groceries & Kebutuhan Pokok': {
            'INDOMARET'           : {'mu': 40000,  'sigma': 25000,  'mcc': '5411', 'w': [7, 6, 5]},
            'ALFAMART'            : {'mu': 40000,  'sigma': 30000,  'mcc': '5411', 'w': [7, 6, 5]},
            'ALFAMIDI'            : {'mu': 60000,  'sigma': 35000,  'mcc': '5411', 'w': [4, 4, 3]},
            'SUPERINDO'           : {'mu': 100000, 'sigma': 120000, 'mcc': '5411', 'w': [3, 4, 2]},
            'HYPERMART'           : {'mu': 200000, 'sigma': 150000, 'mcc': '5411', 'w': [1, 3, 1]},
            'TRANSMART CARREFOUR' : {'mu': 200000, 'sigma': 200000, 'mcc': '5411', 'w': [2, 3, 1]},
            'LOTTEMART'           : {'mu': 200000, 'sigma': 100000, 'mcc': '5411', 'w': [1, 3, 1]},
            'RANCH MARKET'        : {
                'mu': 300000, 'sigma': 200000, 'mcc': '5411', 'w': [0, 1, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Groceries & Kebutuhan Pokok'),  
                    'First Jobber': ('Wants', 'Groceries & Kebutuhan Pokok'), 
                    'Profesional' : ('Needs', 'Groceries & Kebutuhan Pokok'),  
                },
            },
            'FARMERS MARKET'      : {
                'mu': 350000, 'sigma': 180000, 'mcc': '5411', 'w': [0, 2, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Groceries & Kebutuhan Pokok'),
                    'First Jobber': ('Wants', 'Groceries & Kebutuhan Pokok'),
                    'Profesional' : ('Needs', 'Groceries & Kebutuhan Pokok'),
                },
            },
            'PASAR TRADISIONAL'   : {'mu': 80000, 'sigma': 50000,  'mcc': '5411', 'w': [3, 3, 7]},
            'WARUNG SEMBAKO'      : {'mu': 35000,  'sigma': 15000,  'mcc': '5411', 'w': [5, 2, 7]},
            'WATSONS/GUARDIAN'    : {
                'mu': 80000, 'sigma': 50000, 'mcc': '5977', 'w': [5, 3, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Kesehatan & Perawatan Diri'), 
                    'First Jobber': ('Wants', 'Kesehatan & Perawatan Diri'),
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'), 
                },
            },
            'FOTOKOPI & PRINT'    : {'mu': 5000,   'sigma': 2000,   'mcc': '5111', 'w': [5, 4, 5]},
            'GALON & GAS'         : {'mu': 45000,  'sigma': 10000,  'mcc': '5499', 'w': [2, 4, 6]},
            'TOKO ALAT TULIS'     : {'mu': 30000,  'sigma': 15000,  'mcc': '5943', 'w': [4, 3, 4]},
        },
 
        'Kesehatan & Perawatan Diri': {
            'BPJS KESEHATAN'     : {'mu': 150000, 'sigma': 0,      'mcc': '8099', 'w': [3, 5, 6]},
            'APOTEK KIMIA FARMA' : {'mu': 85000,  'sigma': 30000,  'mcc': '5912', 'w': [5, 5, 5]},
            'APOTEK K-24'        : {'mu': 75000,  'sigma': 25000,  'mcc': '5912', 'w': [5, 4, 5]},
            'APOTEK GUARDIAN'    : {'mu': 60000, 'sigma': 40000,  'mcc': '5912', 'w': [4, 3, 1]},
            'KLINIK PRATAMA'     : {'mu': 60000, 'sigma': 40000,  'mcc': '8099', 'w': [3, 4, 4]},
            'PUSKESMAS'          : {'mu': 20000,  'sigma': 5000,   'mcc': '8099', 'w': [5, 2, 6]},
            'RS SWASTA'          : {'mu': 200000, 'sigma': 300000, 'mcc': '8062', 'w': [2, 2, 0]},
            'HALODOC/ALODOKTER'  : {'mu': 60000,  'sigma': 20000,  'mcc': '8099', 'w': [5, 4, 2]},
            'ASURANSI JIWA'      : {
                'mu': 500000, 'sigma': 150000, 'mcc': '6300', 'w': [0, 3, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Hiburan & Langganan'),  
                    'First Jobber': ('Needs', 'Kesehatan & Perawatan Diri'),  
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'),
                },
            },
            'ASURANSI SWASTA'    : {
                'mu': 300000, 'sigma': 80000, 'mcc': '6300', 'w': [0, 3, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Hiburan & Langganan'),
                    'First Jobber': ('Needs', 'Kesehatan & Perawatan Diri'),
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'),
                },
            },
            'FITHUB/GOLD GYM'    : {
                'mu': 300000, 'sigma': 50000, 'mcc': '7997', 'w': [1, 2, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Hiburan & Langganan'),
                    'First Jobber': ('Wants', 'Hiburan & Langganan'),
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'),  
                },
            },
            'CELEBRITY FITNESS'  : {
                'mu': 400000, 'sigma': 80000, 'mcc': '7997', 'w': [0, 1, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Hiburan & Langganan'),
                    'First Jobber': ('Wants', 'Hiburan & Langganan'),
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'),
                },
            },
            'KLINIK KECANTIKAN'  : {
                'mu': 500000, 'sigma': 200000, 'mcc': '7299', 'w': [1, 2, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Belanja Online & Fashion'),  
                    'First Jobber': ('Wants', 'Belanja Online & Fashion'),
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'),
                },
            },
            'SKINCARE ONLINE'    : {
                'mu': 150000, 'sigma': 60000, 'mcc': '5977', 'w': [5, 3, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Belanja Online & Fashion'),
                    'First Jobber': ('Wants', 'Belanja Online & Fashion'),
                    'Profesional' : ('Needs', 'Kesehatan & Perawatan Diri'),
                },
            },
        },
    },
    'Wants': {
        'F&B dan Nongkrong': {
            # --- TIER 3: Fine dining — Wants semua, Pro: Sushi Tei bisa business lunch ---
            'GYU-KAKU AYCE'       : {'mu': 350000, 'sigma': 50000,  'mcc': '5812', 'w': [1, 1, 0]},
            'SHABURI SHABU-SHABU' : {'mu': 200000, 'sigma': 60000,  'mcc': '5812', 'w': [1, 2, 0]},
            'SUSHI TEI'           : {
                'mu': 250000, 'sigma': 80000, 'mcc': '5812', 'w': [1, 2, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Wants', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),  
                },
            },
            'PIZZA HUT'           : {'mu': 120000, 'sigma': 40000,  'mcc': '5812', 'w': [2, 3, 0]},
            'J.CO DONUTS'         : {'mu': 70000,  'sigma': 20000,  'mcc': '5814', 'w': [2, 3, 0]},
            'BREADTALK'           : {'mu': 45000,  'sigma': 15000,  'mcc': '5814', 'w': [4, 3, 1]},
            'BOBA/MINUMAN KEKINIAN': {'mu': 25000, 'sigma': 8000,   'mcc': '5814', 'w': [5, 5, 2]},
            'MIXUE INDONESIA'     : {'mu': 20000,  'sigma': 8000,   'mcc': '5814', 'w': [5, 5, 4]},
            'CAFE AESTHETIC'      : {'mu': 65000,  'sigma': 20000,  'mcc': '5814', 'w': [2, 3, 0]},
            'STARBUCKS'           : {
                'mu': 65000, 'sigma': 20000, 'mcc': '5814', 'w': [2, 3, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),  
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'), 
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),  
                },
            },
            'FORE COFFEE'         : {
                'mu': 30000, 'sigma': 10000, 'mcc': '5814', 'w': [5, 4, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'KOPI KENANGAN'       : {
                'mu': 25000, 'sigma': 10000, 'mcc': '5814', 'w': [5, 5, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'JANJI JIWA'          : {
                'mu': 25000, 'sigma': 8000, 'mcc': '5814', 'w': [4, 5, 2],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'KOPITIAM'            : {
                'mu': 35000, 'sigma': 15000, 'mcc': '5814', 'w': [4, 4, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },

            # --- TIER 2: Makan siang mid-range — Needs FJ & Pro ---
            'MCDONALD\'S'         : {
                'mu': 60000, 'sigma': 20000, 'mcc': '5814', 'w': [5, 5, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'), 
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'KFC INDONESIA'       : {
                'mu': 55000, 'sigma': 20000, 'mcc': '5814', 'w': [5, 5, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'HOKBEN'              : {
                'mu': 60000, 'sigma': 15000, 'mcc': '5812', 'w': [4, 4, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'YOSHINOYA'           : {
                'mu': 55000, 'sigma': 15000, 'mcc': '5812', 'w': [4, 4, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'SOLARIA'             : {
                'mu': 70000, 'sigma': 25000, 'mcc': '5812', 'w': [4, 4, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'PEPPER LUNCH'        : {
                'mu': 120000, 'sigma': 30000, 'mcc': '5812', 'w': [1, 3, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'MIE GACOAN'          : {
                'mu': 30000, 'sigma': 15000, 'mcc': '5814', 'w': [4, 6, 4],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'), 
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),  
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'BAKSO LAPANGAN TEMBAK': {
                'mu': 35000, 'sigma': 10000, 'mcc': '5812', 'w': [3, 5, 4],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'RUMAH MAKAN PADANG'  : {
                'mu': 30000, 'sigma': 10000, 'mcc': '5812', 'w': [3, 5, 6],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),  
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'GOFOOD/GRABFOOD ORDER': {
                'mu': 50000, 'sigma': 20000, 'mcc': '5814', 'w': [5, 5, 2],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'), 
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),  
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'), 
                },
            },
            'SHOPEEFOOD ORDER'    : {
                'mu': 40000, 'sigma': 20000, 'mcc': '5814', 'w': [4, 5, 2],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
 
            # --- TIER 1: Makanan dasar/murah → Needs SEMUA segmen ---
            'NASI PADANG SEDERHANA': {
                'mu': 22000, 'sigma': 5000, 'mcc': '5812', 'w': [6, 4, 7],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'NASI KUNING BU NUNUNG': {
                'mu': 18000, 'sigma': 5000, 'mcc': '5812', 'w': [5, 3, 7],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'WARTEG'              : {
                'mu': 20000, 'sigma': 5000, 'mcc': '5812', 'w': [7, 4, 8],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'WARUNG NASI CAMPUR'  : {
                'mu': 17000, 'sigma': 4000, 'mcc': '5812', 'w': [6, 3, 8],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'BAKSO GEROBAK'       : {
                'mu': 18000, 'sigma': 5000, 'mcc': '5814', 'w': [5, 3, 7],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'MIE AYAM ABANG'      : {
                'mu': 18000, 'sigma': 4000, 'mcc': '5814', 'w': [5, 3, 7],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'INDOMIE REBUS WARUNG': {
                'mu': 7000, 'sigma': 2000, 'mcc': '5814', 'w': [5, 2, 7],
                'gt_seg': {
                    'Mahasiswa'   : ('Needs', 'F&B dan Nongkrong'),
                    'First Jobber': ('Needs', 'F&B dan Nongkrong'),
                    'Profesional' : ('Needs', 'F&B dan Nongkrong'),
                },
            },
            'ES TEH KAMPUS'            : {'mu': 5000,  'sigma': 1000, 'mcc': '5814', 'w': [6, 4, 7]},
            'TEH TARIK PINGGIR JALAN'  : {'mu': 12000, 'sigma': 4000, 'mcc': '5814', 'w': [5, 3, 7]},
            'ES KELAPA/JAJANAN BUAH'   : {'mu': 15000, 'sigma': 5000, 'mcc': '5814', 'w': [1, 3, 7]},
            'JAJANAN GEROBAK'          : {'mu': 10000, 'sigma': 3000, 'mcc': '5814', 'w': [5, 2, 7]},
            'GORENGAN PINGGIR JALAN'   : {'mu': 8000,  'sigma': 2000, 'mcc': '5814', 'w': [6, 2, 8]},
        },
 

        'Hiburan & Langganan': {
            'NETFLIX'          : {
                'mu': 186000, 'sigma': 0, 'mcc': '4899', 'w': [6, 3, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Hiburan & Langganan'),
                    'First Jobber': ('Wants', 'Hiburan & Langganan'),
                    'Profesional' : ('Needs', 'Hiburan & Langganan'), 
                },
            },
            'SPOTIFY'          : {
                'mu': 54900, 'sigma': 0, 'mcc': '4899', 'w': [5, 5, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Hiburan & Langganan'),
                    'First Jobber': ('Needs', 'Hiburan & Langganan'),  
                    'Profesional' : ('Needs', 'Hiburan & Langganan'),
                },
            },
            'DISNEY+ HOTSTAR'  : {'mu': 49000,  'sigma': 0,      'mcc': '4899', 'w': [5, 3, 0]},
            'PRIME VIDEO'      : {'mu': 79000,  'sigma': 0,      'mcc': '4899', 'w': [4, 3, 0]},
            'APPLE MUSIC'      : {'mu': 59000,  'sigma': 0,      'mcc': '4899', 'w': [5, 2, 0]},
            'YOUTUBE PREMIUM'  : {'mu': 59000,  'sigma': 0,      'mcc': '4899', 'w': [4, 4, 1]},
            'VIDIO.COM'        : {'mu': 35000,  'sigma': 0,      'mcc': '4899', 'w': [3, 4, 2]},
            'TIX.ID / KONSER'  : {'mu': 200000, 'sigma': 80000,  'mcc': '7832', 'w': [6, 2, 0]},
            'TIMEZONE'         : {'mu': 150000, 'sigma': 50000,  'mcc': '7999', 'w': [5, 2, 0]},
            'TRANS STUDIO'     : {'mu': 250000, 'sigma': 60000,  'mcc': '7996', 'w': [5, 2, 0]},
            'KARAOKE INUL VIZTA': {'mu': 180000, 'sigma': 60000, 'mcc': '7929', 'w': [5, 2, 0]},
            'GENSHIN IMPACT'   : {'mu': 150000, 'sigma': 60000,  'mcc': '7999', 'w': [5, 3, 0]},
            'CGV CINEMAS'      : {'mu': 60000,  'sigma': 15000,  'mcc': '7832', 'w': [5, 4, 1]},
            'XXI / CINEPLEX 21': {'mu': 55000,  'sigma': 10000,  'mcc': '7832', 'w': [5, 4, 1]},
            'STEAM/VALORANT'   : {'mu': 120000, 'sigma': 50000,  'mcc': '7999', 'w': [5, 4, 1]},
            'PUBG MOBILE UC'   : {'mu': 80000,  'sigma': 30000,  'mcc': '7999', 'w': [4, 3, 1]},
            'MOBILE LEGENDS DM': {'mu': 50000,  'sigma': 20000,  'mcc': '7999', 'w': [4, 4, 2]},
            'FREE FIRE DM'     : {'mu': 30000,  'sigma': 15000,  'mcc': '7999', 'w': [3, 4, 2]},
        },
 
        'Belanja Online & Fashion': {
            'ZARA INDONESIA'     : {
                'mu': 500000, 'sigma': 250000, 'mcc': '5651', 'w': [1, 1, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Belanja Online & Fashion'),
                    'First Jobber': ('Needs', 'Belanja Online & Fashion'),  
                    'Profesional' : ('Needs', 'Belanja Online & Fashion'),  
                },
            },
            'UNIQLO INDONESIA'   : {
                'mu': 450000, 'sigma': 150000, 'mcc': '5651', 'w': [2, 3, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Belanja Online & Fashion'),
                    'First Jobber': ('Needs', 'Belanja Online & Fashion'),
                    'Profesional' : ('Needs', 'Belanja Online & Fashion'),
                },
            },
            'H&M INDONESIA'      : {
                'mu': 300000, 'sigma': 100000, 'mcc': '5651', 'w': [2, 2, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Belanja Online & Fashion'),
                    'First Jobber': ('Needs', 'Belanja Online & Fashion'),
                    'Profesional' : ('Needs', 'Belanja Online & Fashion'),
                },
            },
            'COTTON INK'         : {'mu': 250000, 'sigma': 80000,  'mcc': '5651', 'w': [2, 3, 0]},
            'SOCIOLLA'           : {'mu': 250000, 'sigma': 80000,  'mcc': '5977', 'w': [2, 3, 0]},
            'ZALORA'             : {'mu': 350000, 'sigma': 150000, 'mcc': '5651', 'w': [2, 3, 0]},
            'ERIGO STORE'        : {'mu': 180000, 'sigma': 60000,  'mcc': '5651', 'w': [5, 4, 1]},
            'BRAND LOKAL IG'     : {'mu': 150000, 'sigma': 60000,  'mcc': '5651', 'w': [3, 4, 3]},
            'BLIBLI'             : {'mu': 200000, 'sigma': 100000, 'mcc': '5311', 'w': [4, 4, 2]},
            'TOKOPEDIA'          : {'mu': 150000, 'sigma': 80000,  'mcc': '5311', 'w': [7, 6, 5]},
            'SHOPEE'             : {'mu': 120000, 'sigma': 60000,  'mcc': '5311', 'w': [7, 6, 5]},
            'LAZADA'             : {'mu': 130000, 'sigma': 70000,  'mcc': '5311', 'w': [5, 5, 4]},
            'AKSESORIS HP ONLINE': {'mu': 30000,  'sigma': 15000,  'mcc': '5311', 'w': [5, 4, 3]},
            'THRIFT SHOP IG'     : {'mu': 80000,  'sigma': 30000,  'mcc': '5651', 'w': [6, 4, 6]},
            'PRELOVED CAROUSELL' : {'mu': 60000,  'sigma': 25000,  'mcc': '5651', 'w': [5, 3, 7]},
            'STICKER TELEGRAM/WA': {'mu': 10000,  'sigma': 0,      'mcc': '5734', 'w': [3, 3, 2]},
        },

        'Produktivitas & Digital': {
            'ADOBE CC'       : {
                'mu': 350000, 'sigma': 0, 'mcc': '5734', 'w': [3, 2, 0],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),  
                    'First Jobber': ('Needs', 'Produktivitas & Digital'),  
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'CHATGPT PLUS'   : {
                'mu': 280000, 'sigma': 0, 'mcc': '5734', 'w': [5, 3, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),
                    'First Jobber': ('Needs', 'Produktivitas & Digital'),  
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'FIGMA PRO'      : {
                'mu': 200000, 'sigma': 0, 'mcc': '5734', 'w': [4, 4, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),
                    'First Jobber': ('Needs', 'Produktivitas & Digital'),
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'ZOOM PRO'       : {
                'mu': 215000, 'sigma': 0, 'mcc': '5734', 'w': [3, 3, 1],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),
                    'First Jobber': ('Needs', 'Produktivitas & Digital'), 
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'MICROSOFT 365'  : {
                'mu': 79000, 'sigma': 0, 'mcc': '5734', 'w': [3, 4, 3],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),
                    'First Jobber': ('Needs', 'Produktivitas & Digital'),
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'NOTION'         : {
                'mu': 70000, 'sigma': 0, 'mcc': '5734', 'w': [3, 4, 3],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),
                    'First Jobber': ('Needs', 'Produktivitas & Digital'),
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'CANVA PRO'      : {
                'mu': 50000, 'sigma': 0, 'mcc': '5734', 'w': [6, 5, 3],
                'gt_seg': {
                    'Mahasiswa'   : ('Wants', 'Produktivitas & Digital'),
                    'First Jobber': ('Needs', 'Produktivitas & Digital'),
                    'Profesional' : ('Needs', 'Produktivitas & Digital'),
                },
            },
            'E-READER/WEBTOON': {'mu': 25000, 'sigma': 10000, 'mcc': '5734', 'w': [5, 4, 4]},
            'FOTO STOCK ONLINE': {'mu': 15000, 'sigma': 5000, 'mcc': '5734', 'w': [2, 3, 3]},
        },
    },
 
    'Transfer': {
        'Transfer P2P': {
            'REKENING TEMAN': {'mu': 50000,  'sigma': 30000, 'mcc': '4829', 'w': [5, 5, 4]},
            'OVO/DANA/GOPAY' : {'mu': 100000, 'sigma': 50000, 'mcc': '6540', 'w': [5, 5, 4]},
            'SHOPEEPAY'      : {'mu': 80000,  'sigma': 40000, 'mcc': '6540', 'w': [4, 5, 4]},
            'LINKAJA'        : {'mu': 60000,  'sigma': 30000, 'mcc': '6540', 'w': [3, 4, 4]},
        },
        'Investasi & Finansial': {
            'BIBIT REKSADANA'        : {'mu': 300000,  'sigma': 150000,  'mcc': '6211', 'w': [2, 5, 6]},
            'REKSADANA BAREKSA'      : {'mu': 200000,  'sigma': 80000,   'mcc': '6211', 'w': [2, 4, 6]},
            'AJAIB SEKURITAS'        : {'mu': 800000,  'sigma': 300000,  'mcc': '6211', 'w': [2, 4, 5]},
            'STOCKBIT'               : {'mu': 1200000, 'sigma': 400000,  'mcc': '6211', 'w': [2, 3, 5]},
            'MIRAE ASSET'            : {'mu': 1500000, 'sigma': 500000,  'mcc': '6211', 'w': [2, 3, 4]},
            'PT INDO PREMIER'        : {'mu': 1000000, 'sigma': 300000,  'mcc': '6211', 'w': [2, 3, 4]},
            'TABUNGAN EMAS PEGADAIAN': {'mu': 200000,  'sigma': 50000,   'mcc': '6012', 'w': [2, 4, 6]},
            'EMAS DIGITAL INDOGOLD'  : {'mu': 150000,  'sigma': 60000,   'mcc': '6012', 'w': [2, 4, 6]},
            'DEPOSITO BANK'          : {'mu': 2000000, 'sigma': 500000,  'mcc': '6012', 'w': [1, 3, 6]},
            'JENIUS MAXI SAVER'      : {'mu': 1000000, 'sigma': 300000,  'mcc': '6012', 'w': [1, 4, 6]},
            'ORI/SUKUK RITEL'        : {'mu': 5000000, 'sigma': 1000000, 'mcc': '6211', 'w': [1, 2, 5]},
            'INDODAX'                : {'mu': 500000,  'sigma': 200000,  'mcc': '6211', 'w': [5, 3, 2]},
            'TOKOCRYPTO'             : {'mu': 250000,  'sigma': 100000,  'mcc': '6211', 'w': [5, 3, 2]},
            'KRIPTO PLUANG'          : {'mu': 300000,  'sigma': 120000,  'mcc': '6211', 'w': [5, 3, 2]},
        },
    },
}

print("Master merchant siap.")

Membangun master merchant dictionary...
Master merchant siap.


# Tahap 3 — Mesin Waktu & Generator Transaksi

## Engine Transaksi

Loop 90 hari × 500 nasabah. Per hari per user:
- Gaji masuk setiap tgl 1, plus auto-debit Netflix & BPJS
- Frekuensi per kategori ditentukan `poisson_lambda_per_kategori` × persona aktif
- Dynamic throttling berlaku untuk Unconflicted & Tightwad; Spendthrift dibebaskan
- Windfall: 2% per hari mendapat pemasukan dadakan
- Cause-and-effect: persona turun jika survival streak >= 7 hari (`is_dynamic=1`)
- Output bersih: `training_data_autoencoder.csv` hanya dari `label_anomali==0` & Debit

## Tahap 3 — Running Syntax

In [10]:
print("Menjalankan engine transaksi (v2)...")

tanggal_awal   = datetime(2026, 1, 1)
total_hari     = 90
transaksi_list = []
id_trx_counter = 10001

list_windfall = ['PROFIT TRADING', 'HADIAH/GIFT', 'REFUND BELANJA', 'SIDE HUSTLE PROJECT', 'BONUS APRESIASI']

dict_p2p_ground_truth = {
    "uang kos"        : ("Needs", "Tagihan & Utilitas"),
    "patungan listrik": ("Needs", "Tagihan & Utilitas"),
    "patungan air"    : ("Needs", "Tagihan & Utilitas"),
    "beli token"      : ("Needs", "Tagihan & Utilitas"),
    "kas kelas"       : ("Needs", "Tagihan & Utilitas"),
    "fotokopi modul"  : ("Needs", "Groceries & Kebutuhan Pokok"),
    "bayar buku"      : ("Needs", "Groceries & Kebutuhan Pokok"),
    "bayar gacoan"    : ("Wants", "F&B dan Nongkrong"),
    "kopi tadi"       : ("Wants", "F&B dan Nongkrong"),
    "jajan gofood"    : ("Wants", "F&B dan Nongkrong"),
    "netflix"         : ("Wants", "Hiburan & Langganan"),
    "tiket bioskop"   : ("Wants", "Hiburan & Langganan"),
    "patungan spotify": ("Wants", "Hiburan & Langganan"),
    "utang mabar"     : ("Wants", "Hiburan & Langganan"),
    "hadiah"          : ("Wants", "Belanja Online & Fashion"),
    "beli baju"       : ("Wants", "Belanja Online & Fashion"),
    "sepatu thrifting": ("Wants", "Belanja Online & Fashion"),
    "kado ultah"      : ("Wants", "Belanja Online & Fashion"),
}

# =============================================================================
# 1. Lambda harian per kategori_detail per persona
# =============================================================================
poisson_lambda_per_kategori = {
    'Spendthrift': {
        'F&B dan Nongkrong'           : 4.5,   
        'Transportasi'                : 1.5,   
        'Groceries & Kebutuhan Pokok' : 0.8,   
        'Kesehatan & Perawatan Diri'  : 0.3,   
        'Hiburan & Langganan'         : 1.4,   
        'Belanja Online & Fashion'    : 0.6,   
        'Produktivitas & Digital'     : 0.15, 
        'Tagihan & Utilitas'          : 0.10,  
        'Investasi & Finansial'       : 0.05,  
        'Transfer P2P'                : 0.8,   
    },
    'Unconflicted': {
        'F&B dan Nongkrong'           : 2.5,   
        'Transportasi'                : 1.2,   
        'Groceries & Kebutuhan Pokok' : 0.6,   
        'Kesehatan & Perawatan Diri'  : 0.2,   
        'Hiburan & Langganan'         : 0.8,  
        'Belanja Online & Fashion'    : 0.25, 
        'Produktivitas & Digital'     : 0.10,  
        'Tagihan & Utilitas'          : 0.10,  
        'Investasi & Finansial'       : 0.20, 
        'Transfer P2P'                : 0.6,  
    },
    'Tightwad': {
        'F&B dan Nongkrong'           : 1.7, 
        'Transportasi'                : 1.0,   
        'Groceries & Kebutuhan Pokok' : 2.0,  
        'Kesehatan & Perawatan Diri'  : 0.05, 
        'Hiburan & Langganan'         : 0.08,  
        'Belanja Online & Fashion'    : 0.05, 
        'Produktivitas & Digital'     : 0.07,  
        'Tagihan & Utilitas'          : 0.05, 
        'Investasi & Finansial'       : 0.5,  
        'Transfer P2P'                : 0.10,   
    },
}

KATEGORI_BESAR_MAP = {
    'F&B dan Nongkrong'           : 'Wants',
    'Transportasi'                : 'Needs',
    'Groceries & Kebutuhan Pokok' : 'Needs',
    'Kesehatan & Perawatan Diri'  : 'Needs',
    'Hiburan & Langganan'         : 'Wants',
    'Belanja Online & Fashion'    : 'Wants',
    'Produktivitas & Digital'     : 'Wants',
    'Tagihan & Utilitas'          : 'Needs',
    'Investasi & Finansial'       : 'Transfer',
    'Transfer P2P'                : 'Transfer',
}

# =============================================================================
# Throttling 3-tier:
#   CORE  — kebutuhan dasar, tidak pernah 0 (makan, transport, groceries)
#   SEMI  — dikurangi di Panik, sedikit di Survival (kesehatan, hiburan, P2P, tagihan)
#   HARD  — di-block penuh saat Survival (belanja, produk digital, investasi)
#
# Multiplier per mode [Sultan, Panik, Survival]
# =============================================================================
CORE_CATS = {'F&B dan Nongkrong', 'Transportasi', 'Groceries & Kebutuhan Pokok'}
SEMI_CATS  = {'Kesehatan & Perawatan Diri', 'Hiburan & Langganan', 'Transfer P2P', 'Tagihan & Utilitas'}
HARD_CATS  = {'Belanja Online & Fashion', 'Produktivitas & Digital', 'Investasi & Finansial'}

THROTTLE = {
    'core': [0.9, 0.80, 0.60], 
    'semi': [0.8, 0.55, 0.25],  
    'hard': [0.75, 0.40, 0.00], 
}

PEAK_CATS = {'F&B dan Nongkrong', 'Hiburan & Langganan', 'Belanja Online & Fashion', 'Transfer P2P'}
SEGMEN_IDX = {'Mahasiswa': 0, 'First Jobber': 1, 'Profesional': 2}

sisa_saldo_user = {
    row['id_user']: round(row['gaji_bulanan'] * random.uniform(0.3, 1.2), -3)
    for _, row in df_nasabah.iterrows()
}
survival_streak     = {row['id_user']: 0                    for _, row in df_nasabah.iterrows()}
active_persona_user = {row['id_user']: row['persona_dasar'] for _, row in df_nasabah.iterrows()}
segmen_user         = {row['id_user']: row['segmen_demografi'] for _, row in df_nasabah.iterrows()}
gaji_user           = {row['id_user']: row['gaji_bulanan']     for _, row in df_nasabah.iterrows()}
invest_bulan_ini    = {row['id_user']: 0.0                     for _, row in df_nasabah.iterrows()}


def inject_typo(text):
    if np.random.rand() < 0.15:
        noise_type = random.choice(['drop_char', 'alay_case', 'no_space'])
        if noise_type == 'drop_char' and len(text) > 4:
            idx = random.randint(0, len(text) - 1)
            return text[:idx] + text[idx + 1:]
        elif noise_type == 'alay_case':
            return "".join([c.upper() if random.random() > 0.5 else c.lower() for c in text])
        elif noise_type == 'no_space':
            return text.replace(" ", "")
    return text


def get_realistic_hour(kategori_detail, persona):
    if kategori_detail == 'F&B dan Nongkrong':
        if persona == 'Spendthrift' and np.random.rand() < 0.08:
            return random.randint(22, 23) if random.random() > 0.5 else random.randint(0, 4)
        peaks = [8, 12, 19]
        base  = random.choice(peaks)
        return min(23, max(0, base + random.randint(-1, 2)))
    elif kategori_detail == 'Hiburan & Langganan':
        return random.randint(19, 23)
    elif kategori_detail == 'Tagihan & Utilitas':
        return random.randint(8, 11)
    else:
        return random.randint(9, 21)


def process_cat(user_id, kat_detail, kat_besar, count, current_date, persona, segmen):
    global id_trx_counter
    seg_idx   = SEGMEN_IDX[segmen]
    merchants = list(master_merchant[kat_besar][kat_detail].keys())
    weights   = [master_merchant[kat_besar][kat_detail][m]['w'][seg_idx] for m in merchants]

    # Jika semua bobot 0 untuk segmen ini, fallback ke bobot merata
    if sum(weights) == 0:
        weights = [1] * len(merchants)

    for _ in range(count):
        mrc = random.choices(merchants, weights=weights, k=1)[0]
        pm  = master_merchant[kat_besar][kat_detail][mrc]

        sigma_cap = min(pm['sigma'], pm['mu'] * 0.40) if pm['mu'] > 0 else pm['sigma']
        price     = abs(np.random.normal(pm['mu'], sigma_cap))
        price     = max(2000.0, price)  

        if np.random.rand() < 0.3:
            price += random.randint(1000, 6500)
        anom = 1 if np.random.rand() < 0.02 else 0
        if anom:
            price *= random.uniform(3.0, 8.0)

        # Cap investasi maks 10% gaji/bulan untuk gaji < Rp 5.7 juta
        if kat_detail == 'Investasi & Finansial' and gaji_user[user_id] < 5_729_876:
            batas = gaji_user[user_id] * 0.1
            sisa_kuota = batas - invest_bulan_ini[user_id]
            if sisa_kuota <= 0:
                continue
            price = min(price, sisa_kuota)

        if sisa_saldo_user[user_id] > price + 5000:
            sisa_saldo_user[user_id] -= price
            note      = "-"
            m_final   = mrc
            gt_besar  = kat_besar
            gt_detail = kat_detail

            if 'gt_seg' in pm and segmen in pm['gt_seg']:
                gt_besar, gt_detail = pm['gt_seg'][segmen]

            if pm['mcc'] == '4829' and kat_detail != 'Investasi & Finansial':
                m_final   = f"TRF KE {fake.name().upper()}"
                kata_asli = random.choice(list(dict_p2p_ground_truth.keys()))
                gt_besar  = dict_p2p_ground_truth[kata_asli][0]
                gt_detail = dict_p2p_ground_truth[kata_asli][1]
                note      = inject_typo(kata_asli)

            hour = get_realistic_hour(kat_detail, persona)

            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id,
                current_date.replace(hour=hour, minute=random.randint(0, 59)),
                'Debit', m_final, note, pm['mcc'],
                round(price, 2), round(sisa_saldo_user[user_id], 2),
                kat_besar, kat_detail, anom,
                gt_besar, gt_detail,
            ])
            if kat_detail == 'Investasi & Finansial':
                invest_bulan_ini[user_id] += price
            id_trx_counter += 1


# =============================================================================
# LOOPING UTAMA (90 HARI)
# =============================================================================
for hari in range(total_hari):
    current_date = tanggal_awal + timedelta(days=hari)
    is_weekend   = current_date.weekday() >= 5

    peak_wants_multiplier = 1.0
    if current_date.day <= 5:
        peak_wants_multiplier = 1.3
    elif current_date.day == current_date.month:
        peak_wants_multiplier = 1.4
    if is_weekend:
        peak_wants_multiplier *= 1.2

    if current_date.day == 1:
        for index, row in df_nasabah.iterrows():
            user_id = row['id_user']
            gaji    = row['gaji_bulanan']
            sisa_saldo_user[user_id] += gaji

            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id,
                current_date.replace(hour=7), 'Kredit',
                'TRF MASUK - PENDAPATAN TETAP', "-", '0000', float(gaji),
                round(sisa_saldo_user[user_id], 2),
                'Pemasukan', 'Pendapatan Bulanan', 0, None, None,
            ])
            id_trx_counter += 1

            subs = [
                ('NETFLIX',        'Wants', 'Hiburan & Langganan'),
                ('BPJS KESEHATAN', 'Needs', 'Kesehatan & Perawatan Diri'),
            ]
            for merchant_name, cat_big, cat_dtl in subs:
                p    = master_merchant[cat_big][cat_dtl][merchant_name]
                cost = p['mu']
                if sisa_saldo_user[user_id] > cost:
                    sisa_saldo_user[user_id] -= cost
                    segmen = segmen_user[user_id]
                    gt_big, gt_dtl = (cat_big, cat_dtl)
                    if 'gt_seg' in p and segmen in p['gt_seg']:
                        gt_big, gt_dtl = p['gt_seg'][segmen]
                    transaksi_list.append([
                        f"TRX-{id_trx_counter}", user_id,
                        current_date.replace(hour=9), 'Debit',
                        merchant_name, "Auto-Debit Monthly", p['mcc'], float(cost),
                        round(sisa_saldo_user[user_id], 2),
                        cat_big, cat_dtl, 0, gt_big, gt_dtl,
                    ])
                    id_trx_counter += 1

            if hari > 0 and row['is_dynamic'] == 1 and survival_streak[user_id] >= 7:
                cp = active_persona_user[user_id]
                if cp == 'Spendthrift':
                    active_persona_user[user_id] = 'Unconflicted'
                elif cp == 'Unconflicted':
                    active_persona_user[user_id] = 'Tightwad'
            survival_streak[user_id]  = 0
            invest_bulan_ini[user_id] = 0.0
            df_nasabah.loc[
                df_nasabah['id_user'] == user_id,
                f'persona_bulan_{(hari // 30) + 1}'
            ] = active_persona_user[user_id]

    for index, row in df_nasabah.iterrows():
        user_id = row['id_user']
        gaji    = row['gaji_bulanan']

        if np.random.rand() < 0.02:
            win_val = round(abs(np.random.normal(500000, 300000)), -3)
            sisa_saldo_user[user_id] += win_val
            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id,
                current_date.replace(hour=14), 'Kredit',
                f"TRF MASUK - {random.choice(list_windfall)}", "Rejeki", '0000', float(win_val),
                round(sisa_saldo_user[user_id], 2),
                'Pemasukan', 'Pemasukan Tambahan', 0, None, None,
            ])
            id_trx_counter += 1

        curr_p  = active_persona_user[user_id]
        segmen  = segmen_user[user_id]
        ratio   = sisa_saldo_user[user_id] / gaji

        if ratio <= 0.15:
            survival_streak[user_id] += 1

        # Mode saldo
        if ratio > 0.7:
            mode_idx = 0   # Sultan
        elif ratio > 0.15:
            mode_idx = 1   # Panik
        else:
            mode_idx = 2   # Survival

        for kat_detail, lambda_base in poisson_lambda_per_kategori[curr_p].items():
            kat_besar = KATEGORI_BESAR_MAP[kat_detail]

            if kat_detail in CORE_CATS:
                tier = 'core'
            elif kat_detail in SEMI_CATS:
                tier = 'semi'
            else:
                tier = 'hard'

            throttle = 1.0 if curr_p == 'Spendthrift' else THROTTLE[tier][mode_idx]
            peak = peak_wants_multiplier if kat_detail in PEAK_CATS else 1.0

            lam   = lambda_base * throttle * peak
            count = np.random.poisson(lam)
            if count > 0:
                process_cat(user_id, kat_detail, kat_besar, count, current_date, curr_p, segmen)


df_transaksi = pd.DataFrame(transaksi_list, columns=[
    'id_transaksi', 'id_user', 'timestamp', 'tipe_mutasi', 'deskripsi_mutasi',
    'catatan_mutasi', 'mcc', 'nominal', 'sisa_saldo', 'kategori_besar',
    'kategori_detail', 'label_anomali',
    'gt_kategori_besar', 'gt_kategori_detail',
])
df_transaksi = df_transaksi.sort_values(by=['timestamp', 'id_user']).reset_index(drop=True)
print(f"Selesai. Total transaksi: {len(df_transaksi):,}")
display(df_transaksi.head())


Menjalankan engine transaksi (v2)...
Selesai. Total transaksi: 150,601


,id_transaksi,id_user,timestamp,tipe_mutasi,deskripsi_mutasi,catatan_mutasi,mcc,nominal,sisa_saldo,kategori_besar,kategori_detail,label_anomali,gt_kategori_besar,gt_kategori_detail
0,TRX-11638,USR-409,2026-01-01 00:00:00,Debit,ES TEH KAMPUS,-,5814,5125.29,37366874.71,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
1,TRX-14935,USR-002,2026-01-01 00:04:00,Debit,GORENGAN PINGGIR JALAN,-,5814,10081.93,2226400.42,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
2,TRX-14765,USR-213,2026-01-01 00:04:00,Debit,J.CO DONUTS,-,5814,103463.45,9689432.75,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
3,TRX-12987,USR-455,2026-01-01 00:07:00,Debit,BREADTALK,-,5814,46478.05,28800096.47,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
4,TRX-13198,USR-229,2026-01-01 00:25:00,Debit,JANJI JIWA,-,5814,10348.19,6691040.00,Wants,F&B dan Nongkrong,0,Needs,F&B dan Nongkrong


In [9]:
print("\nDistribusi kategori (Debit):")
print(df_transaksi[df_transaksi['tipe_mutasi']=='Debit']['kategori_detail'].value_counts())
print("\nMinimal nominal (Debit):", df_transaksi[df_transaksi['tipe_mutasi']=='Debit']['nominal'].min())
print("Distribusi label_anomali:", df_transaksi['label_anomali'].value_counts().to_dict())


Distribusi kategori (Debit):
kategori_detail
F&B dan Nongkrong              63321
Transportasi                   24056
Groceries & Kebutuhan Pokok    23507
Hiburan & Langganan            14643
Transfer P2P                    8543
Belanja Online & Fashion        4654
Kesehatan & Perawatan Diri      4142
Investasi & Finansial           2332
Produktivitas & Digital         1486
Tagihan & Utilitas              1167
Name: count, dtype: int64

Minimal nominal (Debit): 42.68
Distribusi label_anomali: {0: 147671, 1: 2603}


In [7]:
display(df_nasabah[['id_user', 'persona_dasar', 'is_dynamic', 'persona_bulan_1', 'persona_bulan_2', 'persona_bulan_3']].head(20))

,id_user,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
0,USR-362,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
1,USR-074,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
2,USR-375,Tightwad,0,Tightwad,Tightwad,Tightwad
3,USR-156,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
4,USR-105,Tightwad,1,Tightwad,Tightwad,Tightwad
5,USR-395,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
6,USR-378,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
7,USR-125,Spendthrift,1,Spendthrift,Tightwad,Spendthrift
8,USR-069,Tightwad,0,Tightwad,Tightwad,Tightwad
9,USR-451,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted


In [8]:
df_transaksi.to_csv(f'{OUTPUT_DIR}/df_transaksi.csv', index=False)
print(f"df_transaksi ({len(df_transaksi):,} baris) disimpan ke:")
print(f"{OUTPUT_DIR}/df_transaksi.csv")


df_training_autoencoder = df_transaksi[
    (df_transaksi['label_anomali'] == 0) &
    (df_transaksi['tipe_mutasi']   == 'Debit')
].copy() 
df_training_autoencoder.to_csv(f'{OUTPUT_DIR}/training_data_autoencoder.csv', index=False)
print(f"training_data_autoencoder ({len(df_training_autoencoder):,} baris) disimpan ke:")
print(f"{OUTPUT_DIR}/training_data_autoencoder.csv")

print(f"\nRasio anomali dalam df_transaksi:")
print(df_transaksi['label_anomali'].value_counts())
print(f"\nKategori detail dalam training set:")
print(df_training_autoencoder['kategori_detail'].value_counts())

df_transaksi (150,274 baris) disimpan ke:
../../../data/new/df_transaksi.csv
training_data_autoencoder (145,248 baris) disimpan ke:
../../../data/new/training_data_autoencoder.csv

Rasio anomali dalam df_transaksi:
label_anomali
0    147671
1      2603
Name: count, dtype: int64

Kategori detail dalam training set:
kategori_detail
F&B dan Nongkrong              62189
Transportasi                   23634
Groceries & Kebutuhan Pokok    23082
Hiburan & Langganan            14390
Transfer P2P                    8384
Belanja Online & Fashion        4579
Kesehatan & Perawatan Diri      4095
Investasi & Finansial           2289
Produktivitas & Digital         1461
Tagihan & Utilitas              1145
Name: count, dtype: int64
